# Building model

Now that we have performed exploratory data analysis (EDA) in a [separate notebook](./understanding_data.ipynb), we can move on to building and testing models.

## Step 0: Basic prep

Get the main input directory.

In [1]:
from pathlib import Path

data_dir = Path('../input/jane-street-real-time-market-data-forecasting')

## Step 1: Data details

- From our preliminary EDA, we found that that 3 features, namely, `feature_09`, `feature_10`, and `feature_11`, are categorical features, with categories represented as numbers, seemingly $\leq$ 20.
    - Note, however, that these were results from the `partition_id=0` training data; we should investigate this for all partitions to make sure we get the correct number of categories per feature
- Similarly, we need to be able to adequately encode the `symbol_id` values, so we have to go through the whole data to make sure we know how many there are
    - Let's not forget to make sure our data processing steps account for the possibility of a new `symbol_id` value, not originally present in the training data. 
- We also observed a number of features filled completely with `NaN` values; these features will be removed. For the other features, imputation with zero (0.0) will be used

### Step 1.0: Scan the data

In [2]:
import polars as pl

training_scan = pl.scan_parquet(data_dir / 'train.parquet')

# Make sure to sort properly
training_scan = training_scan.sort(
    by=['date_id', 'time_id'],
    descending=[False, False]
)

### Step 1.1: Get unique values

Now, let's get the unique values for each entry of interest (including `NaN`)

In [3]:
# Specify the categorical fields
categorical_fields = [f'feature_{i:02d}' for i in [9, 10, 11]] + ['symbol_id']

# Initialize a dictionary to keep the categories
categorical_fields = dict.fromkeys(categorical_fields)

for field in categorical_fields.keys():
    # Get the info
    categories = set(training_scan.select(pl.col(field).unique()).collect()[field].to_list())
    categorical_fields[field] = categories
    print(f"Field {field} has {len(categories)} distinct categories:")
    print(categories)
    print('=' * 120)

Field feature_09 has 22 distinct categories:
{2, 4, 9, 11, 12, 14, 15, 25, 26, 30, 34, 42, 44, 46, 49, 50, 57, 64, 68, 70, 81, 82}
Field feature_10 has 9 distinct categories:
{1, 2, 3, 4, 5, 6, 7, 10, 12}
Field feature_11 has 31 distinct categories:
{388, 261, 9, 522, 11, 13, 16, 150, 534, 24, 25, 410, 539, 158, 159, 34, 40, 297, 171, 48, 50, 59, 62, 63, 66, 195, 76, 336, 214, 230, 376}
Field symbol_id has 39 distinct categories:
{0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38}


### Step 1.2: Recompute `NaN` feature for the entire dataset

Recall several features are fully `NaN` in partition 0; we should ignore those.

Those were:
- feature_00
- feature_21
- feature_02
- feature_03
- feature_04
- feature_01
- feature_31
- feature_27
- feature_26

Let's make sure this trend remains for all partitions first. We will want to make sure that, for the features we keep that have more than 5% `NULL` entries, we create an additional "missing" flag to indicate to the model the value was missing.

In [4]:
# Get the total number of entries
total_num_entries = training_scan.select(pl.len()).collect().item()

# Get the counts
training_null_counts = training_scan.null_count().collect()

# Prepare dictionary to keep the features with more than 5% NULL entries
features_with_non_negligible_null = {}
large_null_fraction_threshold = 0.05

for feature in [f'feature_{i:02d}' for i in range(79)]:
    fraction_null = training_null_counts[feature].item() / total_num_entries
    if fraction_null >= large_null_fraction_threshold:
        print(f'{feature} has {100*fraction_null:.1f}% NULL entries')
        features_with_non_negligible_null[feature] = fraction_null

feature_00 has 6.8% NULL entries
feature_01 has 6.8% NULL entries
feature_02 has 6.8% NULL entries
feature_03 has 6.8% NULL entries
feature_04 has 6.8% NULL entries
feature_21 has 17.9% NULL entries
feature_26 has 17.9% NULL entries
feature_27 has 17.9% NULL entries
feature_31 has 17.9% NULL entries
feature_39 has 9.1% NULL entries
feature_42 has 9.1% NULL entries
feature_50 has 9.0% NULL entries
feature_53 has 9.0% NULL entries


Clearly, our EDA on partition 0 alone was not enough: as we see, the features that were previously fully absent are now present! Therefore, we should keep all the features, but treat these ones carefully.

### Step 1.3: Check time information

Let's make sure that we understand the time information better.

#### Step 1.3.0: Days

Let's make sure the days are all sequential, without any skips.

In [5]:
date_is_sequential = training_scan.select(
    (pl.col('date_id').diff().fill_null(0) <= 1).all()
    ).collect().item()

print(f'Is `date_id` composed of sequential entries? {date_is_sequential}')

Is `date_id` composed of sequential entries? True


Fantastic, that means we do not have to worry about keeping track of skipped days, and can use a simple flag to indicate if we are starting a new day or not.

#### Step 1.3.1: Time IDs

Let's also see if we can get a slightly better understanding of the `time_id` entries.

In [6]:
time_id_range = training_scan.select(pl.col('time_id').min().alias('min_time_id'),
                                     pl.col('time_id').max().alias('max_time_id')).collect()

time_id_range.show()

min_time_id,max_time_id
i16,i16
0,967


This indicates that the `time_id`s are likely minutes: the range is close to 960 = 16 * 60. Market hours are usually from 9:30 AM to 4:00 PM, with pre-market starting at 4:00 AM, and after-market hours ending at 8:00 PM. This is about 16 hours, or, equivalently, 960 minutes. This will be useful for processing these values and normalizing them.

## Step 2: Model details and data processing

Now that we have better information about critical parts of the data for the whole training set, let's create the necesary steps to processes this data to serve as model inputs. For that, we also need to better define what will be the inputs to the model (and the general model architecture).

### Step 2.0: Model details

Seeing as the goal of the data processor is to serve data to the model, we first need to delineate how the model will work. 

Since I am using this as a way to learn, I want to use a recurrent neural network (RNN)-based strategy. It is not something I have done in the past, but it somewhat resembles some of my previous work in Kálmán and particl filtering, and hidden Markov models (HMM). However, one of the primary difficulties with using an out-of-the-box RNN is that, at future prediction times, we may be given some non-zero amount of symbols that were not present during training (and may not be provided features for the symbols we already saw).

To circumvent this issue, I decided to take some inspiration from online forums, and the general idea will be the same: having a hidden state that is used for predictions, and updated from timestep to timestep. However, at a given timestep, the update will be a bit more involved, following these steps:
1. Gather all of the (79) features for all of the known (39) symbols
2. Create "market"-like symbols, which track, feature-by-feature, the daily and timestamp average, standard deviation, minimum, maximum, and count, over all symbols (known and unknown). That means, ten (10) "new" symbols are created, each also containing 79 features (recall some of these will have to be one-hot encoded, so the dimensionality will be larger than 79).
    - <mark>NOTE:</mark> for these operations, missing values should be ignored, rather than imputed.
    - <mark>NOTE:</mark> the new "market"-stats symbols would be `daily_mean_so_far`, `daily_std_so_far`, `daily_min_so_far`, `daily_max_so_far`, `log1p(daily_count_so_far)`, `mean_now`, `std_now`, `min_now`, `max_now`, `log1p(count_now)`
    - <mark>NOTE:</mark> other than the mean, the other statistics are pretty meaningless for the one-hot encodded categories. However, we will keep them to not overly complicated this process.
3. Impute missing features with 0.
4. Prepare the first input to the model, containing 49 symbols (39 known + 10 market statistics) with 79 zero-imputed features each (again, recall the categorical features will increase the actual dimensionality along the feature dimension). Let's call this matrix $M_t$, since it somewhat corresponds to the market at time $t$
    - <mark>NOTE:</mark> critically, $M_t$ is missing exact information of the unknown symbols! However, with the statistics of the current timestamp, this is partially mitigated.
5. $M_t$ is used as input to an attention-head, which is used to encode this information in some latent representation $L_t$
6. The latent embedding $L_t$ is combined with the present timestamp $T_t$, as well as with the previous timestamp $T_{t-1}$ and the difference between the dates, and used to update the hidden state $h_{t-1}$ to $h_t$ through some form of gated recurrent unit (GRU) architecture
7. Each symbol provided is then passed through its own embedding, combined with its 79 features to form another latent represenation of this symbol, which is then combined with the updated hidden state $h_t$ to produce a prediction. This operation would likely be something like `concat([symbol_embedding, features]) --> MLP --> latent_rep --> concat([latent_rep, h_t]) --> MLP --> prediction`

### Step 2.1: Prepare data for a given timestamp

First, let's prepare the functionality for aggregating the data for a single timestamp (that is, a pairing `(date_id, time_id)`).

#### Step 2.1.0: Define necessary market indicators

Recall we wish to keep track of some rolling market indicators. Let's explicitly define them.

In [7]:
from typing import Optional
from dataclasses import dataclass

import torch

@dataclass
class RollingMarketStatsDaily():
    """
    Stores relevant rolling statistics about the market for a given day
    """
    rolling_avg: torch.Tensor  # rolling mean
    rolling_std: torch.Tensor  # rolling standard deviation
    rolling_min: torch.Tensor  # rolling minimum value
    rolling_max: torch.Tensor  # rolling maximum value
    rolling_cnt: torch.Tensor  # rolling count


def combine_statistics(statistics: list[Optional[RollingMarketStatsDaily]]) -> RollingMarketStatsDaily:
    """
    Combines "independent" samples of rolling statistics. Useful for combining present and past statistics

    Args:
        statistics: list of rolling statistics to combine

    Returns:
        combined_statistics: combined statistics
    """
    if statistics[0] is None:
        raise ValueError(f"First entry in the statistics cannot be None!")
    
    if len(statistics) == 1:
        return statistics[0]

    # Initialize what we need to keep track of as we go along
    global_summation = statistics[0].rolling_avg * statistics[0].rolling_cnt
    global_sum_squares = (statistics[0].rolling_std ** 2) * statistics[0].rolling_cnt
    global_min = statistics[0].rolling_min
    global_max = statistics[0].rolling_max
    global_cnt = statistics[0].rolling_cnt

    # Iterate
    for new_stats in statistics[1:]:
        if new_stats is None:
            continue
        global_summation += (new_stats.rolling_avg * new_stats.rolling_cnt)
        global_sum_squares += ((new_stats.rolling_std ** 2) * new_stats.rolling_cnt)
        global_min = torch.min(global_min, new_stats.rolling_min)
        global_max = torch.max(global_max, new_stats.rolling_max)
        global_cnt += new_stats.rolling_cnt
    
    # Compute new mean and std
    global_avg = global_summation / torch.clamp(global_cnt, min=1.)
    global_var = global_sum_squares / torch.clamp(global_cnt, min=1.)
    global_std = torch.sqrt(global_var)

    return RollingMarketStatsDaily(rolling_avg=global_avg,
                                   rolling_std=global_std,
                                   rolling_min=global_min,
                                   rolling_max=global_max,
                                   rolling_cnt=global_cnt)

#### Step 2.1.1: Prepare market snapshot

Finally, we can now define the function that creates the market snapshot.

In [55]:
from typing import Optional


class MarketSnapshotCreator():
    """
    Class to help create market snapshots

    Args:
        symbol_vocabulary: list of integers denoting the symbol IDs considered "known"
        feature_vocabulary: list of strings indicating the names of ALL the features
        categorical_features_info: dictionary where the keys are the categorical features, and the values are lists of
            the different categories present. NOTE that the keys should be a subset of the feature_vocabulary!
        rng_seed: seed for the random number generator
    """
    def __init__(self,
                 symbol_vocabulary: list[int],
                 feature_vocabulary: list[str],
                 categorical_features_info: dict[str, list[int]],
                 rng_seed: int = 1234):
        self.symbol_vocabulary = symbol_vocabulary
        self.feature_vocabulary = feature_vocabulary
        self.categorical_features_info = categorical_features_info
        self.rng = torch.random.manual_seed(rng_seed)

    @property
    def symbol_vocabulary(self) -> list[int]:
        return list(self.symbol_to_index.keys())
    
    @symbol_vocabulary.setter
    def symbol_vocabulary(self, vocab: list[int]) -> None:
        if not isinstance(vocab, list):
            raise TypeError(f"Expected symbol vocabulary to be list, but got {type(vocab)} instead!")
        symbol_to_index = {}
        for i, symbol in enumerate(vocab):
            if not isinstance(symbol, int):
                raise TypeError(f"Symbol in index {i} is not an integer! Symbol = {symbol}")
            symbol_to_index[symbol] = i
        self._symbol_to_index = symbol_to_index

    @property
    def symbol_to_index(self) -> dict[int, int]:
        """
        Provides read-only (copy) for the symbol-to-index mapping
        """
        return self._symbol_to_index.copy()

    @property
    def categorical_features_info(self) -> dict[str, list[int]]:
        return self._categorical_features_info

    @categorical_features_info.setter
    def categorical_features_info(self, info: dict[str, list[int]]):
        # Set the categorical features
        for feat, cats in info.items():
            if feat not in self.feature_vocabulary:
                raise ValueError(f"Feature {feat} not present in feature vocabulary!")
            if not isinstance(cats, list):
                raise TypeError(f"Categories should be lists, but, for feature {feat}, they are {type(cats)}: {cats}")
        self._categorical_features_info = info.copy()

    def create_market_snapshot(
            self,
            timestamp_data: pl.DataFrame,
            rolling_stats: Optional[RollingMarketStatsDaily] = None,
            probability_symbol_unknown: float = 0.0) -> tuple[torch.Tensor, RollingMarketStatsDaily, torch.Tensor]:
        """
        Creates a market snapshot given data for a specific timestamps

        Args:
            timestamp_data: data for the present timestamp to be processed and converted to a numpy array; each row should
                have a symbol_id, and the features should be stored in the columns
            rolling_stats: rolling market statistics; if not provided, will assume the day just started
            probability_symbol_unknown: probability that a known symbol will be considered as "unkown" when assembling
                the snapshot

        Returns:
            market_snapshot: a tensor corresponding to the present snapshot only (excludes rolling stats); its rows
                correspond to the symbols, as well as the avg_now, std_now, min_now, max_now, and cnt_now, and its
                columns are the features. Shape = (num_symbols + 5, num_features)
            updated_rolling_stats: updated rolling market statistics
            unknown_symbol_feats: a tensor containing the features for the unknown (truly or chosen) symbols.
                Shape = (num_unknown, num_features)
        """
        # =================================== Step 0: Gather the data in tensor form ===================================
        tensor_data_dict = timestamp_data.to_torch("dict", label='symbol_id', features=self.feature_vocabulary)
        symbol_ids = tensor_data_dict['label']
        og_features = tensor_data_dict['features']
        num_entries = symbol_ids.shape[0]
        if og_features.shape[0] != num_entries:
            raise ValueError(
                f"Mismatch in dimensions! {num_entries} symbols found, but {og_features.shape[0]} features!")

        # ==================================== Step 1: One-hot categorical features ====================================
        features_list = []  # auxiliary to help with the one-hot encoding
        for i, feature in enumerate(self.feature_vocabulary):
            column = og_features[:, i]
            # If we are dealing with a categorical feature, we need to one-hot
            if feature in self.categorical_features_info.keys():
                # Get the categories
                category_ids = {cat: i for i, cat in enumerate(self.categorical_features_info[feature])}
                # Initialize one-hot tensor
                one_hot = torch.zeros((num_entries, len(category_ids)),
                                      dtype=torch.float16)  # No need for full precision for ones and zeroes
                # Populate one-hot tensor
                for j in range(num_entries):
                    value = column[j].item()
                    if value in category_ids.keys():
                        one_hot[j, category_ids[value]] = 1.0
                # Add it to the list
                features_list.append(one_hot)
            # Otherwise, just transform it into a column and append
            else:
                features_list.append(column.unsqueeze(1))
        # Assemble processed features
        processed_features = torch.cat(features_list, dim=1)

        # ============================= Step 2: Compute current statistics (ignoring NaN) ==============================
        mask = ~torch.isnan(processed_features)  # shape = (num_entries, num_feats)
        # Compute count per feature
        count_now = mask.sum(dim=0).float()  # shape = (num_feats,)
        # Compute mean: sum all elements (replacing `NaN` with zero) and divide by the count
        sum_now = torch.where(mask, processed_features, 0.).sum(dim=0)  # shape = (num_feats,)
        mean_now = sum_now / torch.clamp(count_now, min=1.)  # shape = (num_feats,)
        # Compute standard deviation in similar fashion: compute variance and take the square root
        diff = torch.where(mask, processed_features - mean_now, 0.)  # shape = (num_entries, num_feats)
        var_now = (diff ** 2).sum(dim=0) / torch.clamp(count_now, min=1.)  # shape = (num_feats,)
        std_now = torch.sqrt(var_now)
        # Compute minimum and maximum
        min_now = torch.where(mask, processed_features, float('inf')).min(dim=0).values
        max_now = torch.where(mask, processed_features, float('-inf')).max(dim=0).values
        # At the end, if a feature is fully missing, the min/max will be positive/negative infinity, so we want to
        # correct that!
        missing_feat = count_now == 0
        min_now = torch.where(~missing_feat, min_now, 0.)
        max_now = torch.where(~missing_feat, max_now, 0.)

        # ===================================== Step 3: Update rolling statistics ======================================
        statistics_now = RollingMarketStatsDaily(rolling_avg=mean_now,
                                                 rolling_std=std_now,
                                                 rolling_min=min_now,
                                                 rolling_max=max_now,
                                                 rolling_cnt=count_now)
        updated_statistics = combine_statistics(statistics=[statistics_now, rolling_stats])

        # ========================================== Step 4: Zero-inputation ===========================================
        processed_features = torch.where(mask, processed_features, 0.)

        # ================= Step 5: Assemble symbols tensor (recall to check for "unknow" conversion) ==================
        num_symbols = len(self.symbol_vocabulary)
        num_feats_total = processed_features.shape[1]
        snapshot = torch.zeros((num_symbols, num_feats_total), dtype=torch.float32)

        # Recall to store the features from the unknown symbols!
        unknown_symbol_features = []

        for i in range(num_entries):
            symbol = int(symbol_ids[i].item())
            # If symbol not accounted for, or if we roll a probability smaller than expected, skip
            if (symbol not in self.symbol_vocabulary) or \
                (torch.rand(1, generator=self.rng) < probability_symbol_unknown):
                unknown_symbol_features.append(processed_features[i, :])
                continue
            # Otherwise, add to snapshot
            snapshot[self.symbol_to_index[symbol], :] = processed_features[i, :]
        
        # Make the unknown symbol features a tensor
        if len(unknown_symbol_features) > 0:
            unknown_symbol_features = torch.stack(unknown_symbol_features, dim=0)
        else:
            unknown_symbol_features = torch.empty(size=(0, num_feats_total))

        # ================================== Step 6: Append general market statistics ==================================
        stats_rows = torch.stack([mean_now,
                                  std_now,
                                  min_now,
                                  max_now,
                                  count_now], dim=0)
        snapshot = torch.cat([snapshot, stats_rows], dim=0)

        return snapshot, updated_statistics, unknown_symbol_features

In [57]:
# Create a market snapshot creator object
test_snapshot_creator = MarketSnapshotCreator(
    symbol_vocabulary=list(categorical_fields['symbol_id']),
    feature_vocabulary=[f'feature_{feat:02d}' for feat in range(79)],
    categorical_features_info={k: list(v) for k, v in categorical_fields.items() if k != 'symbol_id'})

# Get test data
test_date_id = 37
test_time_id = 420
test_data_0 = training_scan.filter((pl.col('date_id') == test_date_id) & (pl.col('time_id') == test_time_id)).collect()

# Check the returns
processed_feats, updated_stats, unkown_symbol_features = \
    test_snapshot_creator.create_market_snapshot(timestamp_data=test_data_0, probability_symbol_unknown=0.)

# compute the expected number of features
expected_num_feats = 79
for feature, cats in categorical_fields.items():
    if feature == 'symbol_id':
        continue
    expected_num_feats += len(cats) - 1  # add num of categories from one-hot encoding, remove 1 from "continuous"

# Print
print(f"Expected number of features = {expected_num_feats}")
print(f"Total number of symbols = {test_data_0.shape[0]}")
print(f"Processed features shape = {processed_feats.shape}")
print(f"Rolling_avg.shape = {updated_stats.rolling_avg.shape}")
print(f"Rolling_std.shape = {updated_stats.rolling_std.shape}")
print(f"Rolling_min.shape = {updated_stats.rolling_min.shape}")
print(f"Rolling_max.shape = {updated_stats.rolling_max.shape}")
print(f"Rolling_cnt.shape = {updated_stats.rolling_cnt.shape}")
print(f"Any `NaN`? {torch.any(torch.isnan(processed_feats))}")
print(f'Unknown symbol features shape = {unkown_symbol_features.shape}')

Expected number of features = 138
Total number of symbols = 11
Processed features shape = torch.Size([44, 138])
Rolling_avg.shape = torch.Size([138])
Rolling_std.shape = torch.Size([138])
Rolling_min.shape = torch.Size([138])
Rolling_max.shape = torch.Size([138])
Rolling_cnt.shape = torch.Size([138])
Any `NaN`? False
Unknown symbol features shape = torch.Size([0, 138])


In [58]:
# Now, try the next one
test_data_1 = training_scan.filter(
    (pl.col('date_id') == test_date_id) & (pl.col('time_id') == test_time_id + 1)).collect()

# Check the returns
processed_feats_1, updated_stats_1, unkown_symbol_features = \
    test_snapshot_creator.create_market_snapshot(timestamp_data=test_data_1,
                                                 rolling_stats=updated_stats,
                                                 probability_symbol_unknown=1.)

# Print
print(f"Expected number of features = {expected_num_feats}")
print(f"Total number of symbols = {test_data_1.shape[0]}")
print(f"Processed features shape = {processed_feats_1.shape}")
print(f"Rolling_avg.shape = {updated_stats_1.rolling_avg.shape}")
print(f"Rolling_std.shape = {updated_stats_1.rolling_std.shape}")
print(f"Rolling_min.shape = {updated_stats_1.rolling_min.shape}")
print(f"Rolling_max.shape = {updated_stats_1.rolling_max.shape}")
print(f"Rolling_cnt.shape = {updated_stats_1.rolling_cnt.shape}")
print(f"Any `NaN`? {torch.any(torch.isnan(processed_feats_1))}")
print(f'Unknown symbol features shape = {unkown_symbol_features.shape}')

Expected number of features = 138
Total number of symbols = 11
Processed features shape = torch.Size([44, 138])
Rolling_avg.shape = torch.Size([138])
Rolling_std.shape = torch.Size([138])
Rolling_min.shape = torch.Size([138])
Rolling_max.shape = torch.Size([138])
Rolling_cnt.shape = torch.Size([138])
Any `NaN`? False
Unknown symbol features shape = torch.Size([11, 138])


### Step 2.2: Establish model architecture

In [59]:
import torch.nn as nn


class AttentionGRUMarketModel(nn.Module):
    """
    Definition of the main architecture for the market model we have.

    It employs a transformer to encode the features from the known symbols plus market indicators as a latent
    representation, which is used as the input to a gated recurrent unit (GRU) that updates the hidden state. The hidden
    state is then used to make predictions for each symbol

    Args:
        num_feats_total: total number of features after post-processing (one-hot encoding, etc.)
        num_known_symbols: total number of known symbols
        num_targets: number of targets/responders to predict
        hidden_dim: dimensionality of the hidden vector. Defaults to 128.
        market_latent_dim: dimensionality of the latent space used to represent the present market conditions. Defaults
            to 128.
        symbol_id_embed_dim: dimensionality of the embeddings for the symbol IDs. Defaults to 16.
        pre_decoder_dim: dimensionality of the latent space prior to the "decoder" side. Defaults to 128.
        inner_decoder_dim: dimensionality of the latent space in the decoder. Defaults to 128.
        post_decoder_dim: dimensionality after the decoder, but prior to the last linear layer. Defaults to 128.
        num_attention_heads: number of attention heads to use for encoding latent representation of market state.
            Defaults to 4.
        num_attention_layers: number of attention layers to stack when embedding present market data. Defaults to 2.
        attention_dropout: dropout rate for attention layers. Defaults to 0.1.
        transformer_dim_feedforward: dimensionality of the feedforward network model. Defaults to 256.
    """
    def __init__(
        self,
        num_feats_total: int,
        num_known_symbols: int,
        num_targets: int,
        hidden_dim: int = 128,
        market_latent_dim: int = 218,
        symbol_id_embed_dim: int = 16,
        pre_decoder_dim: int = 128,
        inner_decoder_dim: int = 128,
        post_decoder_dim: int = 128,
        num_attention_heads: int = 4,
        num_attention_layers: int = 2,
        attention_dropout: float = 0.1,
        transformer_dim_feedforward: int = 256) -> None:
        # Init the super for convenience
        super().__init__()

        # Store the more critical info
        self.num_feats_total = num_feats_total
        self.num_known_symbols = num_known_symbols
        self.num_targets = num_targets
        self.hidden_dim = hidden_dim
        self.symbol_id_embed_dim = symbol_id_embed_dim

        # --- Encode features for transformer ---
        self.pre_attention_encoder = nn.Linear(num_feats_total, market_latent_dim)

        # --- Attention encoder ---
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=market_latent_dim,
            nhead=num_attention_heads,
            dim_feedforward=transformer_dim_feedforward,
            dropout=attention_dropout,
            batch_first=True
        )
        self.market_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_attention_layers)

        # --- GRU ---
        self.gru = nn.GRUCell(market_latent_dim + 2,  # accounting for time_id and new_day flag
                              hidden_dim)

        # --- Symbol embedding ---
        self.sym_emb = nn.Embedding(num_known_symbols + 1,  # need to add one for the catch-all "unknown" symbol_id
                                    symbol_id_embed_dim)

        # --- Feature encoder ---
        self.decoder_embedding = nn.Sequential(
            nn.Linear(num_feats_total + symbol_id_embed_dim, pre_decoder_dim),
            nn.ReLU(),
            nn.Linear(pre_decoder_dim, inner_decoder_dim),
            nn.ReLU()
        )

        # --- Final head ---
        self.final_predictor = nn.Sequential(
            nn.Linear(inner_decoder_dim + hidden_dim + 1,  # include `time_id` again
                      post_decoder_dim),
            nn.ReLU(),
            nn.Linear(post_decoder_dim, num_targets)
        )

    def forward_step(self,
                     M_t: torch.Tensor,
                     sym_ids: torch.Tensor,
                     features: torch.Tensor,
                     time_id: torch.Tensor,
                     new_day_flag: torch.Tensor,
                     h: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Forward operation.

        Args:
            M_t: representation of current market conditions, including features for every known symbol_id, as well as
                present market statistics and rolling market statistics.
                Shape: (num_known_symbols + 2 * num_market_stats, self.num_feats_total)
            symbol_ids: processed symbol_ids in the range of [0, self.num_known_symbols].
                Shape: (num_observed_symbols,)
            features: processed features.
                Shape: (num_observed_symbols, self.num_feats_total)
            time_id: normalized values of time_id.
                Shape: (1,)
            new_day_flag: scalar (0/1) flag to indicate a new day.
                Shape: (1,)
            h: hidden state at the beginning of each sequence.
                Shape: (self.hidden_dim,)
        """

        # --- Encode market ---
        x = self.pre_attention_encoder(M_t)  # tensor (S_know + market_stats, market_latent_dim)
        x = self.encoder(x)  # tensor (S_know + market_stats, market_latent_dim)
        L_t = x.mean(dim=0).squeeze(0)  # tensor (market_latent_dim,)

        # --- Time + day info ---
        gru_in = torch.cat([L_t, time_id, new_day_flag], dim=-1)  # tensor (market_latent_dim + 2,)

        # --- Update hidden state ---
        h_new = self.gru(gru_in.unsqueeze(0), h.unsqueeze(0)).squeeze(0)  # tensor (hidden_dim,)

        # --- Per-symbol prediction ---
        sym_e = self.sym_emb(sym_ids)  # tensor (num_observed_symbols, symbol_id_embed_dim)
        z = torch.cat([features, sym_e], dim=-1)  # tensor (num_observed_symbols, num_feats_total + symbol_id_embed_dim)

        # Prior to seeing hidden state
        z = self.decoder_embedding(z)  # tensor (num_observed_symbols, inner_decoder_dim)

        # Prepare for concatenation
        h_new_tiled = torch.tile(h_new.unsqueeze(0), [z.shape[0], 1])  # tensor (num_observed_symbols, hidden_dim)
        time_id_tiled = torch.tile(time_id.unsqueeze(0), [z.shape[0], 1])  # tensor (num_observed_symbols, 1)

        # Concatenate
        z = torch.cat([z, h_new_tiled, time_id_tiled],
                      dim=-1)  # tensor (num_observed_symbols, inner_decoder_dim + hidden_dim + 1)

        # Predict
        y = self.final_predictor(z)  # tensor (num_observed_symbols, num_targets)

        return y, h

### Step 2.3: Prepare data processor

Let's now define the processor that converts data into the format(s) expected by the model.

In [ ]:
class MarketDataProcessor(MarketSnapshotCreator):
    """
    Class to prepare data for fitting model

    Args:
        lazyframe: polars LazyFrame of the data
        categorical_features: list of features to be considered categorical
        batch_size: number of sequences that will be batched together for training. Defaults to 10.
        days_per_batch: number of days in each sequence of the batch. Defaults to 14.
        rng_seed: seed for the random number generator
    """
    def __init__(self,
                 lazyframe: pl.LazyFrame,
                 categorical_features: list[str] = [f'feature_{i:02d}' for i in [9, 10, 11]],
                 batch_size: int = 10,
                 days_per_batch: int = 14,
                 rng_seed: int = 1234):
        
        # Static info just for reference
        self._symbol_id_col = 'symbol_id'
        self._time_id_col = 'time_id'
        self._date_id_col = 'date_id'

        # Process the data
        symbol_vocab, categorical_features_info = self.set_lazyframe(lazyframe=lazyframe,
                                                                     categorical_features=categorical_features)
        
        # Initialize parent class
        super().__init__(symbol_vocabulary=symbol_vocab,
                         feature_vocabulary=[f'feature_{i:02d}' for i in range(79)],
                         categorical_features_info=categorical_features_info,
                         rng_seed=rng_seed)
        
        # Store other relevant infor
        self.batch_size = batch_size
        self.days_per_batch = days_per_batch

    def set_lazyframe(self,
                      lazyframe: pl.LazyFrame,
                      categorical_features: list[str]) -> tuple[list[int], dict[str, list[int]]]:
        """
        Processes the lazyframe and outputs the known symbol vocabulary, as well as the categories for the provided
        categorical features.

        Args:
            lazyframe: lazyframe of the data
            categorical_features: list of categorical features
        """
        # Specify the colums we want to get
        cols_of_interest = categorical_features + [self._symbol_id_col]

        # Initialize a dictionary to keep the categories
        categorical_fields = dict.fromkeys(cols_of_interest)

        # Obtain the information
        for field in categorical_fields.keys():
            # Get the info
            categories = set(lazyframe.select(pl.col(field).unique()).collect()[field].to_list())
            categorical_fields[field] = list(categories)

        # Extract symbol vocabular
        symbol_vocab = categorical_fields.pop(self._symbol_id_col)

        # Compute relevant information
        date_id_range = lazyframe.select(
            pl.col(self._date_id_col).min().alias('min_date'),
            pl.col(self._date_id_col).max().alias('max_date')).collect()
        self._max_date = date_id_range['max_date'].item()
        self._min_date = date_id_range['min_date'].item()

        # Keep the lazyframe
        self._lazyframe = lazyframe.sort(
            by=[self._date_id_col, self._time_id_col],
            descending=[False, False]
        )

        # Return desired info
        return symbol_vocab, categorical_fields


In [63]:
time_id_range['max_time_id'].item()

967

## Check backend for Mac

In [9]:
print(torch.backends.mps.is_built())      # Should be True
print(torch.backends.mps.is_available())  # Should be True 

True
True
